# Person 2 + Person 3: Labeling, Clustering, Modeling, and SHAP


In [1]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

for package_name, module_name in [("shap", "shap"), ("scikit-fuzzy", "skfuzzy")]:
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

PROJECT_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / "src").exists()), Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd

from src.clustering import plot_cluster_comparison, run_fcm_exploration
from src.explainability import init_explainer, run_shap_sanity_check, save_global_shap_summary, save_individual_waterfall
from src.model import train_svm
from src.predict import load_prediction_artifacts, predict_patient
from src.risk_labeling import apply_risk_labels


In [2]:
cleaned_df = pd.read_csv("outputs/cleaned_healthcare.csv")
cleaned_df.head()


,TIMESTAMP,HOSPITAL_ID,PATIENT_ID,age,gender,admission_type,medical_condition,heart_rate,systolic_bp,diastolic_bp,...,hour_sin,hour_cos,day_sin,day_cos,severity_score,heart_rate_lag1,oxygen_saturation_lag1,systolic_bp_lag1,gender_encoded,admission_type_encoded
0,2022-01-01 12:09:44,HOSP_01,PAT_000001,3,Male,Emergency,Respiratory,103,90,63,...,1.224647e-16,-1.000000,0.201299,0.97953,1,103.0,92.0,90.0,1,1
1,2022-01-01 09:13:20,HOSP_01,PAT_000002,66,Male,Urgent,Injury,96,128,80,...,7.071068e-01,-0.707107,0.201299,0.97953,0,96.0,98.0,128.0,1,2
2,2022-01-01 02:00:41,HOSP_01,PAT_000003,81,Male,Emergency,Cardiac,99,139,89,...,5.000000e-01,0.866025,0.201299,0.97953,0,99.0,95.0,139.0,1,1
3,2022-01-01 07:42:58,HOSP_01,PAT_000004,62,Female,Urgent,Respiratory,107,128,87,...,9.659258e-01,-0.258819,0.201299,0.97953,1,107.0,95.0,128.0,0,2
4,2022-01-01 22:00:02,HOSP_01,PAT_000005,59,Male,Emergency,Respiratory,89,134,77,...,-5.000000e-01,0.866025,0.201299,0.97953,0,89.0,93.0,134.0,1,1


In [3]:
person2_df, label_distribution = apply_risk_labels(cleaned_df, save_output=False)
person2_df[["risk_score", "risk_category"]].head()


Risk class distribution:
risk_category
Low       17647
Medium     9961
High        665
Name: count, dtype: int64


,risk_score,risk_category
0,1,Medium
1,0,Low
2,0,Low
3,1,Medium
4,0,Low


In [4]:
person2_df, clustering_artifacts = run_fcm_exploration(person2_df, save_outputs=True)
cluster_plot_path = plot_cluster_comparison(person2_df)
print(f"Labeled data saved to: {clustering_artifacts['labeled_data_path']}")
print(f"Cluster plot saved to: {cluster_plot_path}")


FCM completed with FPC=0.7020
Borderline patients (< 0.60 max membership): 3690


Labeled data saved to: D:\Desktop\intelligent Project\outputs\labeled_healthcare.csv
Cluster plot saved to: D:\Desktop\intelligent Project\outputs\cluster_plot.png


In [5]:
labeled_df = pd.read_csv(clustering_artifacts["labeled_data_path"])
training_artifacts = train_svm(labeled_df, save_artifacts=True)
model = training_artifacts["model"]
scaler = training_artifacts["scaler"]
feature_columns = training_artifacts["feature_columns"]
X_train = training_artifacts["X_train"]
X_test = training_artifacts["X_test"]


Model class distribution:
risk_category
Low       17647
Medium     9961
High        665
Name: count, dtype: int64


Accuracy: 0.9786
F1-score (weighted): 0.9786

Classification report:
              precision    recall  f1-score   support

         Low       0.99      0.98      0.99      3530
      Medium       0.96      0.98      0.97      1992
        High       0.97      0.88      0.92       133

    accuracy                           0.98      5655
   macro avg       0.97      0.95      0.96      5655
weighted avg       0.98      0.98      0.98      5655



In [6]:
explainer = init_explainer(model, scaler, training_artifacts["shap_background"], feature_columns)
shap_summary_path = save_global_shap_summary(
    explainer,
    X_test.sample(min(50, len(X_test)), random_state=42),
    list(model.classes_),
    nsamples=50,
)
sample_patient = X_test.iloc[[0]]
predicted_label = model.predict(scaler.transform(sample_patient))[0]
shap_waterfall_path = save_individual_waterfall(
    explainer,
    sample_patient,
    predicted_label,
    list(model.classes_),
    nsamples=50,
)
shap_check = run_shap_sanity_check(explainer, sample_patient, list(model.classes_), nsamples=25)
print(f"SHAP summary saved to: {shap_summary_path}")
print(f"SHAP waterfall saved to: {shap_waterfall_path}")
shap_check


Initialized SHAP explainer once with 75 background samples.


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

SHAP working test passed
{'rows_explained': 1, 'feature_count': 6, 'class_count': 3, 'status': 'SHAP working test passed'}
SHAP summary saved to: D:\Desktop\intelligent Project\outputs\shap_summary.png
SHAP waterfall saved to: D:\Desktop\intelligent Project\outputs\shap_waterfall.png


{'rows_explained': 1,
 'feature_count': 6,
 'class_count': 3,
 'status': 'SHAP working test passed'}

In [7]:
load_prediction_artifacts(
    model=model,
    scaler=scaler,
    explainer=explainer,
    background_data=training_artifacts["shap_background"],
    feature_columns=feature_columns,
)

test_patient = X_test.iloc[0].to_dict()
prediction = predict_patient(test_patient, nsamples=50)
prediction


  0%|          | 0/1 [00:00<?, ?it/s]

{'condition': 'Low',
 'probability': 0.9999998999999635,
 'shap_explanation': {'heart_rate': 0.13442712431495413,
  'systolic_bp': 0.08578060660768805,
  'temperature': 0.03392283675565528,
  'oxygen_saturation': 0.028928340078847332,
  'diastolic_bp': 0.016962931597044657,
  'wait_time': 0.002293984535368887},
 'all_probabilities': {'High': 1.6666674432768123e-14,
  'Low': 0.9999998999999635,
  'Medium': 1.0000001994734589e-07}}

In [8]:
from pathlib import Path

required_outputs = [
    Path("outputs/labeled_healthcare.csv"),
    Path("outputs/scaler.pkl"),
    Path("outputs/svm_model.pkl"),
    Path("outputs/confusion_matrix.png"),
    Path("outputs/shap_summary.png"),
]

print("Person 2 + Person 3 pipeline complete.")
print()
print("Class distribution (Low / Medium / High):")
print(training_artifacts["class_distribution"])
print()
print(f"Accuracy: {training_artifacts['accuracy']:.4f}")
print(f"F1-score (weighted): {training_artifacts['f1_weighted']:.4f}")
print()
print("Confirmation of no leakage:")
print("Person 2 used cleaned unscaled data, and the only scaler is outputs/scaler.pkl fitted on X_train.")
print()
print("Generated output files:")
for path in required_outputs:
    print(path, "OK" if path.exists() else "MISSING")
print()
print("SHAP working confirmation:")
print(shap_check["status"])


Person 2 + Person 3 pipeline complete.

Class distribution (Low / Medium / High):
risk_category
Low       17647
Medium     9961
High        665
Name: count, dtype: int64

Accuracy: 0.9786
F1-score (weighted): 0.9786

Confirmation of no leakage:
Person 2 used cleaned unscaled data, and the only scaler is outputs/scaler.pkl fitted on X_train.

Generated output files:
outputs\labeled_healthcare.csv OK
outputs\scaler.pkl OK
outputs\svm_model.pkl OK
outputs\confusion_matrix.png OK
outputs\shap_summary.png OK

SHAP working confirmation:
SHAP working test passed
